In [1]:
%%capture
%run '/Users/katbutler/MetabolicCost/metCostMod-Result.ipynb'

In [3]:
# KINETIC ENERGY IND_DF
analyzed = set()
dfCount = 1
for subject in result.keys():
    indShiftDict = {}
    if subject[0] in analyzed:
        continue
    analyzed.add(subject[0])
    for activity in result.keys():
    
        if activity[1] in indShiftDict:
            continue
            
        if activity[1] not in result[subject[0]]:
            continue

        ind = result[subject[0]][activity[1]]['APDM_Accel']['Data'][:, code]
        indShift = np.where((ind[:-1] - ind[1:]) != 0)[0]
        
        
        indShiftDict[activity[1]] = indShift

    # Convert to DataFrame for the subject
    indDFKe = pd.DataFrame(dict([(key, pd.Series(value)) for key, value in indShiftDict.items()]))
    indDFKe = indDFKe.fillna(0)
    
    if dfCount <= 10:
        if dfCount < 10:
            exec(f"Subject0{dfCount} = indDFKe")
            dfCount += 1
        else:
            exec(f"Subject{dfCount} = indDFKe")
            dfCount += 1
    
# print(Subject01)

In [ ]:
# KINETIC ENERGY
# 5x 1-minute average KE per interval
analyzed = set()
subjectName = []
index = -1
df_list = []
df_list_R = []

for subject in result.keys():
    # Skip certain activities
    if subject[0] == "Subject02" and subject[1] in ['Backwards','Backwards2']:
        continue
    if subject[0] == "Subject05" and subject[1] in ['Cycling','Cycling2']:
        continue
    kePoints = []
    eePoints = []
    # Initialize/reset DataFrames for each subject
    KE_df = pd.DataFrame()
    KE_means_df = pd.DataFrame(columns=["Segment", "KE_Mean"])
    
    mNorm = mass[index] / 82.2
    
    if subject[0] not in analyzed:
        index += 1
    # print(f"{subject[0]} {subject[1]}\n-----------")
    analyzed.add(subject[0])
    
    indDF = globals()[subject[0]].astype(int)

    # Parameters for different segments
    segments = [
        {'Name': 'W', 'I': 0.02654, 'columns': [5, 6, 7]},    # W
        {'Name': 'C', 'I': 0.33201, 'columns': [14, 15, 16]}, # C
        {'Name': 'AL', 'I': 0.06397, 'columns': [23, 24, 25]}, # AL
        {'Name': 'AR', 'I': 0.06271, 'columns': [32, 33, 34]}  # AR
    ]

    # Calculate kinetic energy for each segment
    for segment in segments:
        segment_name = segment['Name']
        I = segment['I'] * mNorm
        cols = segment['columns']
        
        # Extract and compute omega
        x, y, z = [np.nan_to_num(result[subject[0]][subject[1]]['APDM_Accel']['Data'][:, c], nan=0.0) for c in cols]
        omega = np.sqrt(x**2 + y**2 + z**2)
        KE = 0.5 * I * omega**2
        KE_df[segment_name] = KE

    # Calculate total kinetic energy
    KE_df['total'] = (KE_df['W'] + KE_df['C'] + KE_df['AL'] + KE_df['AR'])
    num_intervals = len(indDF[subject[1]]) - 1

    for i in range(num_intervals):
        interval_start = int(indDF[subject[1]][i])
        interval_end = int(indDF[subject[1]][i + 1])
        samples_per_minute = 60 * 128

        for minute in range(5):
            bin_start = interval_start + minute * samples_per_minute
            bin_end = bin_start + samples_per_minute

            if bin_end <= interval_end:
                mean_val = round(KE_df['total'].iloc[bin_start:bin_end].mean(), 4)
                kePoints.append(mean_val)
            else:
                break

    kePoints = [x for x in kePoints if x == x] # Remove NaN
    # print(kePoints)

    # Energy expenditure calculations
    timeF = result[subject[0]][subject[1]]['APDM_Accel']['Data'][:, 0]
    ke_slice_start = result[subject[0]][subject[1]]['APDM_Accel']['Data'][0, 0] + indDF[subject[1]][0] / 128
    ke_slice_end = result[subject[0]][subject[1]]['APDM_Accel']['Data'][0, 0] + indDF[subject[1]][5] / 128

    VO2 = result[subject[0]][subject[1]]['Metabolics_System']['Data'][:, vo2]
    VCO2 = result[subject[0]][subject[1]]['Metabolics_System']['Data'][:, vco2]
    EE = (16.58 * VO2 + 4.15 * VCO2) / mass[0]
    EE_ground = np.zeros(np.size(EE))

    ind = result[subject[0]][subject[1]]['Metabolics_System']['Data'][:, code]
    ind = ind[:-1] - ind[1:]
    bool_ind = ind != 0.0
    indShift = np.where(bool_ind)[0]
    indShift = np.concatenate((indShift, [len(ind)]))

    T_first_phase = result[subject[0]][subject[1]]['Metabolics_System']['Data'][indShift[0], 0]
    timeM3_first = T_first_phase - (3 * 60)
    bool_time_first = result[subject[0]][subject[1]]['Metabolics_System']['Data'][:, 0] < timeM3_first
    threeMin_first = indShift[0] - np.max(np.where(bool_time_first)[0])

    EE_avg_first = np.mean(EE[indShift[0] - threeMin_first:indShift[0]])

    for i in range(len(indShift)):
        T = result[subject[0]][subject[1]]['Metabolics_System']['Data'][indShift[i], 0]
        timeM3 = T - (3 * 60)
        bool_time = result[subject[0]][subject[1]]['Metabolics_System']['Data'][:, 0] < timeM3
        threeMin = indShift[i] - np.max(np.where(bool_time)[0])

        if i == 0:
            EE_ground[0:indShift[i]] = np.mean(EE[indShift[i] - threeMin:indShift[i]]) - EE_avg_first
            eePoints.append(round(np.mean(EE[indShift[i] - threeMin:indShift[i]] - EE_avg_first), 4))
        elif i == len(indShift) - 1:
            EE_ground[indShift[i - 1]:] = np.mean(EE[-threeMin:]) - EE_avg_first
            eePoints.append(round(np.mean(EE[-threeMin:] - EE_avg_first), 4))
        else:
            EE_ground[indShift[i - 1]:indShift[i] + 1] = np.mean(EE[indShift[i] - threeMin:indShift[i]]) - EE_avg_first
            eePoints.append(round(np.mean(EE[indShift[i] - threeMin:indShift[i]] - EE_avg_first), 4))

    if subject[1] == 'Incline':
        eePoints = eePoints[1:-1]
    if subject[1] != 'Incline':
        eePoints = eePoints[1:-1]

    if len(eePoints) * 5 != len(kePoints):  
        print("Error in " + subject[0] + " " + subject[1])
        continue
    # print(eePoints)

    # For plot
    lastNZ = indDF[subject[1]][indDF[subject[1]] != 0].index[-1]
    time_slice = timeF[indDF[subject[1]][0]:indDF[subject[1]][lastNZ]]
    KE_slice = KE_df['total'][indDF[subject[1]][0]:indDF[subject[1]][lastNZ]]

    ke_eeDF = pd.DataFrame({'ke': kePoints})

    ke_eeDF['ee'] = np.repeat(eePoints, 5)[:len(ke_eeDF)]
    ke_eeDF['Time'] = [360 + 60 * i + 60 * ((i) // 5) for i in range(len(kePoints))]

    newRow = ke_eeDF.iloc[-1].to_dict()
    newRow["Time"] = indDF[subject[1]][len(ke_eeDF) / 5] / 128
    ke_eeDF = pd.concat([ke_eeDF, pd.DataFrame([newRow])], ignore_index=True)
    ke_eeDF['Subject'] = subject[0]
    ke_eeDF['Activity'] = subject[1]
    # print(ke_eeDF)

    # fig, ax1 = plt.subplots()
    # color = 'tab:blue'
    # ax1.set_xlabel('Time (s)')
    # ax1.set_ylabel('Kinetic Energy (W/kg)', color=color)
    # ax1.plot(time_slice, KE_slice, color=color)
    # ax1.tick_params(axis='y', labelcolor=color)

    x = ke_eeDF['Time']
    y = ke_eeDF['ke']
    # plt.step(x, y, where='post', color='black')
    # # for i in range(len(x) - 1):
    # #     plt.text(x[i] + 175, y[i] + 0.03, f'{y[i]}', ha='center', va='bottom')

    # ax2 = ax1.twinx()
    # color = 'tab:orange'
    # ax2.set_ylabel('EE Ground (W/kg)', color=color)
    # ax2.plot(result[subject[0]][subject[1]]['Metabolics_System']['Data'][:, 0], EE_ground, '-', color=color)
    # ax2.tick_params(axis='y', labelcolor=color)

    # plt.xlabel('Time (s)')
    # plt.title('Kinetic Energy vs. EE Ground')
    # plt.show()

    if subject[1] == 'Incline':
        # ke_eeDF.drop(2, inplace=True)
        df_list.append(ke_eeDF)
    else:
        df_list.append(ke_eeDF)

    df_list_R.append(ke_eeDF[:-1])


pd.set_option('display.max_rows', 2000)
final_df_r = pd.concat(df_list_R, ignore_index=True)
print(final_df_r)

In [25]:
# HEART RATE IND_DF
analyzedhr = set()
dfCount = 1
for subject in result.keys():
    indShiftDict = {}
    if subject[0] in analyzedhr:
        continue
    analyzedhr.add(subject[0])
    
    for activity in result.keys():  
        if activity[1] in ['Backwards2', 'Cycling2'] or activity[1] in indShiftDict:
            continue
    
        ind = result['Subject03'][activity[1]]['Metabolics_System']['Data'][:, code]
        indShift = np.where((ind[:-1] - ind[1:]) != 0)[0]
    
        indShiftDict[activity[1]] = indShift

    # Convert to DataFrame for the subject
    indDFhr = pd.DataFrame(dict([(key, pd.Series(value)) for key, value in indShiftDict.items()]))
    indDFhr = indDF.fillna(0)
    
    if dfCount <= 10:
        if dfCount < 10:
            exec(f"Subject0{dfCount} = indDFhr")
            dfCount += 1
        else:
            exec(f"Subject{dfCount} = indDFhr")
            dfCount += 1
    
print(Subject04)


   Backwards  Cycling  Incline  Running  Walking  Stairs
0      129.0    119.0      117    104.0    110.0   129.0
1      276.0    255.0      261    231.0    250.0   283.0
2      419.0    413.0      412    390.0    399.0   455.0
3        0.0    564.0      535    604.0    542.0   640.0
4        0.0    724.0      702    795.0      0.0     0.0
5        0.0      0.0      849      0.0      0.0     0.0


In [26]:
# HEART RATE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_list = []

for subject in result.keys():
    hrPoints = []
    eePoints = []
    
    # Get subject dataframe - handle potential missing subjects
    try:
        indDF = globals()[subject[0]].astype(int)
    except KeyError:
        # print(f"Warning: {subject[0]} not found in globals, skipping...")
        continue

    # Skip irrelevant activities
    if subject[1] in ['Backwards'] and subject[0] == 'Subject02':
        continue
    if subject[1] in ['Cycling'] and subject[0] == 'Subject05':
        continue
    if subject[1] in ['Backwards2', 'Cycling2']:
        continue

    # print(f"{subject[0]} {subject[1]}\n-----------")

    # Extract HR data
    hr = result[subject[0]][subject[1]]['Metabolics_System']['Data'][:, 8]
    time_col = result[subject[0]][subject[1]]['Metabolics_System']['Data'][:, 0]
    hr_df = pd.DataFrame(hr, columns=['HR']).fillna(0)

    # Compute standing average HR from minute 3 to 6
    standing_mask = (time_col >= 3 * 60) & (time_col <= 6 * 60)
    standing_avg_hr = hr_df['HR'][standing_mask].mean()
    # print("Standing Average HR:", standing_avg_hr)
    
    # Process HR data in 1-minute bins
    for i in range(len(indDF[subject[1]]) - 1):
        time_start = int(indDF[subject[1]][i])
        # time_start = result[subject[0]][subject[1]]['Metabolics_System']['Data'][interval_start, 0]
        time_end = int(indDF[subject[1]][i + 1])
        # time_end = result[subject[0]][subject[1]]['Metabolics_System']['Data'][interval_end, 0]
        data = result[subject[0]][subject[1]]['Metabolics_System']['Data']
        
        for minute in range(5):
            bin_start = int(time_start + minute * 60)
            bin_end = int(bin_start + 60)
            
            # Check if bin is within interval
            if bin_end <= time_end:
                # Find indices for time range
                start_mask = data[:, 0] >= bin_start
                end_mask = data[:, 0] < bin_end
                bin_mask = start_mask & end_mask
                
                if np.any(bin_mask):
                    mean_val = round(np.mean(data[bin_mask, 8]) - standing_avg_hr, 4)
                    hrPoints.append(mean_val)
                    
            else:
                break
                
    # Remove NaN values
    hrPoints = [x for x in hrPoints if not np.isnan(x)]
    # print(f"HR Points: {len(hrPoints)} values")

    # if len(hrPoints) == 0:
    #     print(f"No HR data for {subject[0]} {subject[1]}, skipping...")
    #     continue

    # Create aligned dataframe
    min_length = (len(hrPoints))
    hrPoints = hrPoints[:min_length]
    # ee_repeated = np.repeat(eePoints, 5)[:min_length]

    hr_eeDF = pd.DataFrame({
        'HR': hrPoints,
        # 'EE': ee_repeated,
        'Index': list(range(min_length)),
        'Subject': [subject[0]] * min_length,
        'Activity': [subject[1]] * min_length,
        'Time': [360 + 60 * i + 60 * ((i) // 5) for i in range(min_length)]
    })
        
    # print(f"Final dataframe shape: {hr_eeDF.shape}")
    # print(hr_eeDF.head())
    # if subject[1] == 'Incline':
    #     hr_eeDF = hr_eeDF[:-1]
    df_list.append(hr_eeDF)
    
    # Create visualization
    x = hr_eeDF['Time'].to_list()
    y_hr = hr_eeDF['HR'].to_list()
    # y_ee = hr_eeDF['EE'].to_list()

    # fig, ax1 = plt.subplots(figsize=(10, 6))

    # color = 'tab:blue'
    # ax1.set_xlabel('Time (s)')
    # ax1.set_ylabel('HR (bpm)', color=color)
    # ax1.step(x, y_hr, where='post', color=color, label='HR (bpm)', linewidth=2)
    # ax1.tick_params(axis='y', labelcolor=color)

    # ax2 = ax1.twinx()

    # # color = 'tab:orange'
    # # ax2.set_ylabel('EE Ground (W/kg)', color=color)
    # # ax2.step(x, y_ee, where='post', color=color, label='EE Ground', linewidth=2)
    # # ax2.tick_params(axis='y', labelcolor=color)

    # title = f"{subject[0]}, {subject[1]} - HR vs. EE Ground"
    # plt.title(title)

    # # Add legends
    # ax1.legend(loc='upper left')
    # ax2.legend(loc='upper right')

    # fig.tight_layout()
    # plt.show()

# Combine all dataframes
if df_list:
    final_df = pd.concat(df_list, ignore_index=True)
    # print(f"\nFinal combined dataframe shape: {final_df.shape}")
    # print(final_df.head())
# else:
#     print("No data processed successfully.")

pd.set_option('display.max_rows', 2000)
final_df_r = pd.concat(df_list, ignore_index=True)
print(final_df_r)

          HR  Index    Subject   Activity  Time
0     2.2687      0  Subject01  Backwards   360
1    -1.5617      1  Subject01  Backwards   420
2     0.4354      2  Subject01  Backwards   480
3     6.7687      3  Subject01  Backwards   540
4    -0.1329      0  Subject01    Cycling   360
5    -0.6217      1  Subject01    Cycling   420
6    -0.6467      2  Subject01    Cycling   480
7     4.0449      3  Subject01    Cycling   540
8    31.3116      4  Subject01    Cycling   600
9    34.2964      5  Subject01    Cycling   720
10   37.7510      6  Subject01    Cycling   780
11   37.2964      7  Subject01    Cycling   840
12   -3.5776      0  Subject01    Incline   360
13   -1.4665      1  Subject01    Incline   420
14    1.2231      2  Subject01    Incline   480
15    0.8113      3  Subject01    Incline   540
16   29.0745      4  Subject01    Incline   600
17   29.0494      5  Subject01    Incline   720
18   31.6534      6  Subject01    Incline   780
19   31.4304      7  Subject01    Inclin